# PDB Fetch Tools

![PDB Fetch Tools](https://proto-bio.github.io/proto-assets/images/tool/pdb/hero.png)

This notebook demonstrates the two PDB fetch functions provided by proto_tools. `run_pdb_fetch_entry` retrieves entry-level metadata from the RCSB REST API, returning the structure title, experimental method, and resolution for a given accession. `run_pdb_fetch_fasta` fetches the FASTA sequences for all polymer chains in a structure and automatically classifies each chain as protein or nucleotide based on its composition. Together these functions support common protein design workflows where you need to retrieve a structural template and its associated sequences before running inverse folding, mutagenesis, or structure prediction.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pdb")
display_overview("pdb")
display_docs_section("pdb", "Background")

# PDB

The RCSB Protein Data Bank is a database of experimentally determined three-dimensional structures of proteins, nucleic acids, and their complexes. This toolkit provides two CPU-only tools: `pdb-fetch-entry` retrieves structure metadata (title, experimental method, and resolution) for a PDB accession, and `pdb-fetch-fasta` retrieves the chain sequences of an entry and classifies each as protein or nucleic acid. It runs on CPU and requires only network access.

The Protein Data Bank ([Berman et al., 2000](https://doi.org/10.1093/nar/28.1.235)) is the single worldwide archive of experimentally determined macromolecular structures, served here through the RCSB PDB. It is operated by the Research Collaboratory for Structural Bioinformatics (RCSB) at Rutgers University and the University of California San Diego, with funding from the National Science Foundation, the National Institutes of Health, and the Department of Energy. Entries are solved by X-ray crystallography, cryo-electron microscopy, nuclear magnetic resonance spectroscopy, and other experimental methods.

The tools call two RCSB HTTP endpoints directly. `pdb-fetch-entry` issues a GET request to the RCSB Data API core entry endpoint (`https://data.rcsb.org/rest/v1/core/entry/{pdb_id}`) and reads the structure title from `struct.title`, the experimental method from the first `exptl` record, and the resolution from `rcsb_entry_info.resolution_combined`, which covers both X-ray and cryo-EM entries; entries solved by NMR have no resolution value. `pdb-fetch-fasta` requests the FASTA endpoint (`https://www.rcsb.org/fasta/entry/{pdb_id}`), parses each record, extracts the author-assigned chain identifiers from the header, and classifies a sequence as protein when it contains amino-acid letters that do not also occur in nucleotide alphabets. Both tools uppercase the supplied accession, retry transient HTTP failures with backoff, and return an empty result when the accession is not found (HTTP 404). Results reflect the live archive at query time rather than a fixed release snapshot.

## Available tools

In [2]:
display_available_tools("pdb")

- **`run_pdb_fetch_entry()`** — Fetch structure metadata (title, method, resolution) from RCSB PDB
- **`run_pdb_fetch_fasta()`** — Fetch chain sequences from RCSB PDB with protein/nucleotide classification

### `run_pdb_fetch_entry`

`run_pdb_fetch_entry` queries the RCSB REST API to retrieve catalog-level information about a PDB structure: the structure title, experimental method (e.g., X-RAY DIFFRACTION, SOLUTION NMR), and resolution in angstroms. This is useful as a quick lookup step before downloading full structure files, allowing you to confirm the experimental quality and method of a candidate template. We use PDB entry 1LBG, the lac repressor bound to a symmetric operator DNA, as the running example.

In [3]:
from proto_tools import (
    PdbFetchConfig,
    PdbFetchEntryInput,
    run_pdb_fetch_entry,
)

In [4]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_fetch_entry")

# Example input
inputs = PdbFetchEntryInput(pdb_id="1EFA")

**Input** — `PdbFetchEntryInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pdb_id</code> | <code>str</code> | required | PDB accession (e.g. '1LBG') |

In [5]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_fetch_entry")

# Config is shared across both PDB fetch functions | see docs above for all fields
config = PdbFetchConfig()

**Config** — `PdbFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |

In [6]:
# Run the entry fetch
entry_output = run_pdb_fetch_entry(inputs, config)

In [7]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_fetch_entry")

# Inspect the structure metadata
print(f"Title: {entry_output.title}")
print(f"Method: {entry_output.method}")
print(f"Resolution: {entry_output.resolution} A")
print(f"Source: {entry_output.source_url}")

**Output** — `PdbFetchEntryOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>title</code> | <code>str &#124; None</code> | <code>None</code> | Human-readable RCSB entry title |
| <code>method</code> | <code>str &#124; None</code> | <code>None</code> | PDB experimental method (e.g. X-RAY DIFFRACTION, SOLUTION NMR, ELECTRON MICROSCOPY) |
| <code>resolution</code> | <code>float &#124; None</code> | <code>None</code> | Resolution in Å (None for NMR or fiber diffraction) |
| <code>source_url</code> | <code>str &#124; None</code> | <code>None</code> | Request URL |

Title: CRYSTAL STRUCTURE OF THE LAC REPRESSOR DIMER BOUND TO OPERATOR AND THE ANTI-INDUCER ONPF
Method: X-RAY DIFFRACTION
Resolution: 2.6 A
Source: https://data.rcsb.org/rest/v1/core/entry/1EFA


### `run_pdb_fetch_fasta`

`run_pdb_fetch_fasta` retrieves the polymer chain sequences from RCSB PDB in FASTA format and automatically classifies each chain as protein or nucleotide based on its amino acid composition. A PDB structure can contain multiple chains of different types, so this function is useful when you need to extract specific chains for downstream analysis, such as feeding a protein chain into a structure prediction model or aligning it against homologous sequences.

In [8]:
from proto_tools import (
    PdbFetchFastaInput,
    run_pdb_fetch_fasta,
)

In [9]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_fetch_fasta")

# Fetch chain sequences for the lac repressor structure
fasta_inputs = PdbFetchFastaInput(pdb_id="1EFA")

**Input** — `PdbFetchFastaInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pdb_id</code> | <code>str</code> | required | PDB accession (e.g. '1LBG') |

In [10]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_fetch_fasta")

# Config is shared across both PDB fetch functions | see docs above for all fields
fasta_config = PdbFetchConfig()

**Config** — `PdbFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |

In [11]:
# Run the FASTA fetch
fasta_output = run_pdb_fetch_fasta(fasta_inputs, fasta_config)

In [12]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_fetch_fasta")

# Inspect the chain sequences and their classifications
print(f"Chains: {len(fasta_output.chains)}")
print()

for chain in fasta_output.chains:
    chain_type = "protein" if chain.is_protein else "nucleotide"
    preview = chain.sequence[:50] + ("..." if len(chain.sequence) > 50 else "")
    print(f"  Chains {chain.chain_ids}: {chain_type}, length={len(chain.sequence)}")
    print(f"    {preview}")

**Output** — `PdbFetchFastaOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>chains</code> | <code>list[PdbChain]</code> | <code>[]</code> | Parsed chain sequences |
| <code>source_url</code> | <code>str &#124; None</code> | <code>None</code> | Request URL |

**`PdbChain`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>chain_ids</code> | <code>list[str]</code> | <code>[]</code> | Author-assigned chain IDs sharing this sequence |
| <code>header</code> | <code>str</code> | required | FASTA header |
| <code>sequence</code> | <code>str</code> | required | Chain sequence |
| <code>is_protein</code> | <code>bool</code> | required | True if protein, False if nucleic acid |

Chains: 2

  Chains ['D', 'E']: nucleotide, length=21
    GAATTGTGAGCGCTCACAATT
  Chains ['A', 'B', 'C']: protein, length=333
    MKPVTLYDVAEYAGVSYQTVSRVVNQASHVSAKTREKVEAAMAELNYIPN...


## Export Results

Fetched results can be saved to JSON format for downstream use in other tools or analysis pipelines.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

fasta_output.export("pdb_fasta_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'pdb_fasta_results.json'}")

Exported to example_output/pdb_fasta_results.json
